# Giustificazioni verificabili e ritiro su LUBM

Primo benchmark confrontabile tra modelli: **6 query di sottoclasse inizialmente entailed**, tratte
da `owlapy_generated_qa.json`, due per profondità di prova esplicita 1, 2 e 3.
Il dataset è congelato in `benchmarks/subclass_grounding_v1/` e non dipende dalle risposte dei modelli.
HermiT verifica nuovamente ogni caso prima del run. Il vecchio notebook esplorativo resta invariato.

Per ciascuna query il modello riceve:

1. **Intera ontologia originale** (319 triple), query e schema di risposta.
2. **Intera ontologia dopo il ritiro** (318 triple), con la richiesta iniziale e la propria risposta
   autentica nella cronologia. Un singolo assioma necessario è stato rimosso; HermiT conferma
   che la query non è più derivabile. Il modello non riceve risultato atteso, prova gold o feedback.

Le coppie sono indipendenti: non si accumula cronologia tra query. Tutti i casi originali sono
positivi; i negativi sono esclusivamente le corrispondenti versioni dopo il ritiro.

## Ambiente e avvio

Java nel PATH, Ollama avviato e un modello già installato. Kernel **Python (epistemic pilot)**.

```powershell
.\.venv\Scripts\python.exe -m pip install -r requirements-belief-revision-full.txt
```

Eseguire tutte le celle. Ogni modello richiede **12 chiamate** e crea un nuovo run; non ci sono
retry selettivi. Per confrontare altri modelli modificare `MODELS`, mantenendo dataset, prompt e opzioni.
Nessun modello viene scaricato automaticamente.

In [1]:
from pathlib import Path
import json
import sys
from IPython.display import display, Markdown

roots = [Path.cwd(), Path.cwd() / "neurosymbolic-epistemic", Path.cwd().parent]
ROOT = next((p.resolve() for p in roots if (p / "subclass_grounding_benchmark.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Aprire il notebook dalla repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from subclass_grounding_benchmark import DATASET, OPTIONS, load_dataset, run_benchmark, compare_runs, message

MODELS = ["qwen2.5-coder:7b"]  # aggiungere solo nomi esatti di modelli locali
data, graphs, texts, oracle, dataset_hash = load_dataset(DATASET)
print("Dataset:", data["version"], "| SHA-256:", dataset_hash)
print("Casi:", len(data["cases"]), "| opzioni fisse:", OPTIONS)
table = "| Caso | Query | Passaggi nella prova esplicita | Assioma ritirato |\n|---|---|---|---|"
for case in data["cases"]:
    short = lambda uri: uri.split("#")[-1]
    removed = case["removed_axioms"][0]
    table += f"\n| {case['id']} | {short(case['query'][0])} ⊑ {short(case['query'][1])} | {case['explicit_proof_depth']} | {short(removed['subject'])} ⊑ {short(removed['object'])} |"
display(Markdown(table))

Dataset: subclass_grounding_v1 | SHA-256: 1e2fe67ce94eb8aa0c2d2319997f7885c377943eb58275c2cdcaf1a972a49a1f
Casi: 6 | opzioni fisse: {'temperature': 0, 'seed': 42, 'num_ctx': 32768, 'num_predict': 768}


| Caso | Query | Passaggi nella prova esplicita | Assioma ritirato |
|---|---|---|---|
| subclass_01 | Course ⊑ Work | 1 | Course ⊑ Work |
| subclass_02 | Faculty ⊑ Employee | 1 | Faculty ⊑ Employee |
| subclass_03 | Professor ⊑ Employee | 2 | Faculty ⊑ Employee |
| subclass_04 | TechnicalReport ⊑ Publication | 2 | Article ⊑ Publication |
| subclass_05 | AssistantProfessor ⊑ Employee | 3 | Faculty ⊑ Employee |
| subclass_06 | AssociateProfessor ⊑ Employee | 3 | Faculty ⊑ Employee |

## Output richiesto e verifica formale

Il modello restituisce soltanto `answer` (`entailed` / `not_entailed`) e `axioms`:

```json
{
  "answer": "entailed",
  "axioms": [
    {
      "subject": "http://swat.cse.lehigh.edu/onto/univ-bench.owl#Course",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "object": "http://swat.cse.lehigh.edu/onto/univ-bench.owl#Work"
    }
  ]
}
```

L'esempio sopra illustra il formato al lettore del notebook; non è inserito nei prompt.
Con `not_entailed`, `axioms` deve essere `[]`.

Per una risposta positiva con insieme di assiomi citati (J), snapshot (O) e query (q),
verifichiamo:

- correttezza della risposta rispetto a HermiT su (O);
- **provenienza:** (J \subseteq O), quindi ogni assioma è realmente asserito nello snapshot;
- **sufficienza:** (J \models q), tramite HermiT sul solo supporto citato;
- minimalità per inclusione, registrata separatamente e non richiesta per superare il caso.

Una prova alternativa valida passa: non facciamo exact match con la prova di riferimento.
Citare una conclusione inferita come se fosse un assioma asserito non passa.

Per una risposta negativa, la lista vuota è una convenzione di output e non una prova:
il non-entailment è verificato sullo snapshot completo. Nessun testo libero viene richiesto,
quindi questo esperimento misura il supporto strutturato dichiarato e non la coerenza di una spiegazione narrativa.

## Score

Per ogni coppia (i):

- (G_i=1) se la risposta originale è corretta e il suo supporto è presente e sufficiente; altrimenti 0.
- (R_i=1) se dopo il ritiro la risposta è `not_entailed` con lista vuota; altrimenti 0.
- (P_i=G_iR_i): successo sull'intera coppia.

Riportiamo **grounded score** (\sum G_i/N), **withdrawal score** (\sum R_i/N),
e **paired grounded revision score** (\sum P_i/N). Quest'ultimo è lo score congiunto principale.
Il denominatore è sempre costituito da tutte le 6 query: non escludiamo le baseline sbagliate.
Riportiamo anche l'accuracy iniziale, le risposte iniziali corrette prive di supporto valido
e l'eventuale riuso di assiomi ritirati.

Errori di formato del modello contano come insuccessi. Errori infrastrutturali, di contesto o
chiamate mancanti rendono il run **non confrontabile**, anziché abbassare arbitrariamente lo score.
I dettagli e gli insuccessi restano nei log. La finestra fissa è 32768 token; il budget conservativo
e la capacità riportata da Ollama vengono controllati senza tagliare il contesto.

In [2]:
# Ispezionare questa variabile per vedere l'intero prompt, senza chiamare il modello.
example_request = message(data["cases"][0], "original", texts)
print("Caratteri nel primo messaggio:", len(example_request["content"]))
print("Triple nello snapshot originale:", len(graphs["ontology_original.ttl"]))

Caratteri nel primo messaggio: 11772
Triple nello snapshot originale: 319


In [3]:
run_dirs = []
results = {}
for model in MODELS:
    print("\nMODELLO:", model, flush=True)
    directory, result = run_benchmark(model, DATASET)
    run_dirs.append(directory)
    results[model] = result


MODELLO: qwen2.5-coder:7b


subclass_01 original {'answer': 'entailed', 'axioms': [{'object': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Work', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'subject': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Course'}]}


subclass_01 withdrawal {'answer': 'not_entailed', 'axioms': []}


subclass_02 original {'answer': 'entailed', 'axioms': [{'object': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Employee', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'subject': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Faculty'}]}


subclass_02 withdrawal {'answer': 'not_entailed', 'axioms': []}


...

subclass_05 withdrawal {'answer': 'entailed', 'axioms': [{'object': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Professor', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'subject': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#AssistantProfessor'}, {'object': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Faculty', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'subject': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Professor'}, {'object': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Employee', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'subject': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Faculty'}]}


subclass_06 original {'answer': 'not_entailed', 'axioms': []}


subclass_06 withdrawal {'answer': 'entailed', 'axioms': [{'object': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Professor', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'subject': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#AssociateProfessor'}, {'object': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Faculty', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'subject': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Professor'}, {'object': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Employee', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'subject': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Faculty'}]}


{
  "n_pairs": 6,
  "n_responses": 12,
  "complete_for_comparison": true,
  "infrastructure_or_context_errors": 0,
  "format_errors": 0,
  "pair_success_count": 3,
  "paired_grounded_revision_score": 0.5,
  "original_grounded_score": 0.6666666666666666,
  "withdrawal_score": 0.5,
  "original_answer_accuracy": 0.6666666666666666,
  "correct_original_without_valid_support_count": 0,
  "withdrawal_stale_citation_count": 3
}


Run: C:\Users\manud\Desktop\neurosymbolic - epistemic\neurosymbolic-epistemic\generated\subclass_grounding\20260929T125224Z_cbd1a24f


In [4]:
for model, result in results.items():
    display(Markdown(f"### {model}"))
    summary = result["summary"]
    display(Markdown(
        "| Misura | Valore |\n|---|---|\n" + "\n".join(
            f"| {key} | {summary[key]} |" for key in (
                "complete_for_comparison", "paired_grounded_revision_score", "original_grounded_score",
                "withdrawal_score", "original_answer_accuracy", "correct_original_without_valid_support_count"))))
    table = "| Caso | Originale giustificato | Ritiro corretto | Coppia riuscita |\n|---|---|---|---|"
    for pair in result["pairs"]:
        table += f"\n| {pair['case_id']} | {pair['original_success']} | {pair['withdrawal_success']} | {pair['pair_success']} |"
    display(Markdown(table))
    for row in result["responses"]:
        print(row["case_id"], row["condition"], "risposta:", row["answer"],
              "atteso:", row["gold"], "supporto sufficiente:", row["support_sufficient"])
print("Cartelle dei risultati:")
for directory in run_dirs:
    print(directory)

### qwen2.5-coder:7b

| Misura | Valore |
|---|---|
| complete_for_comparison | True |
| paired_grounded_revision_score | 0.5 |
| original_grounded_score | 0.6666666666666666 |
| withdrawal_score | 0.5 |
| original_answer_accuracy | 0.6666666666666666 |
| correct_original_without_valid_support_count | 0 |

| Caso | Originale giustificato | Ritiro corretto | Coppia riuscita |
|---|---|---|---|
| subclass_01 | True | True | True |
| subclass_02 | True | True | True |
| subclass_03 | True | True | True |
| subclass_04 | True | False | False |
| subclass_05 | False | False | False |
| subclass_06 | False | False | False |

subclass_01 original risposta: entailed atteso: entailed supporto sufficiente: True
subclass_01 withdrawal risposta: not_entailed atteso: not_entailed supporto sufficiente: None
subclass_02 original risposta: entailed atteso: entailed supporto sufficiente: True
subclass_02 withdrawal risposta: not_entailed atteso: not_entailed supporto sufficiente: None
subclass_03 original risposta: entailed atteso: entailed supporto sufficiente: True
subclass_03 withdrawal risposta: not_entailed atteso: not_entailed supporto sufficiente: None
subclass_04 original risposta: entailed atteso: entailed supporto sufficiente: True
subclass_04 withdrawal risposta: entailed atteso: not_entailed supporto sufficiente: False
subclass_05 original risposta: not_entailed atteso: entailed supporto sufficiente: None
subclass_05 withdrawal risposta: entailed atteso: not_entailed supporto sufficiente: False
subclass_06 original risposta: not_entailed atteso: entailed supporto sufficiente: None
subclass_06 withdrawal r

## Confrontare i modelli e leggere i limiti

`compare_runs` accetta solo run completi con lo stesso fingerprint di dataset, istruzioni, schema,
opzioni ed evaluator. Si possono aggiungere cartelle di run precedenti alla lista della prossima cella.
Digest del modello, versione Ollama, Java, librerie e piattaforma sono registrati nel manifest.
Per confronti sperimentali controllati mantenere anche ambiente e hardware costanti, o documentare le differenze.

Il campione è un **pilot di sei query**, non una graduatoria affidabile. Le query condividono
parte degli assiomi e alcune ritirano lo stesso collegamento: non sono sei prove indipendenti.
Le query che richiedono assiomi complessi (restrizioni, equivalenze) senza una prova di sole inclusioni
nominate sono fuori dallo scope di questa prima versione, pur restando quei costrutti nel contesto.

In questo protocollo tutti i secondi turni sono negativi. Un modello che impara semplicemente a
cambiare risposta può quindi ottenere un alto score: serviranno controlli senza variazione di entailment
per una misura più forte della revisione selettiva. Qui manteniamo deliberatamente il primo test semplice
richiesto. Non interpretiamo una prova valida come evidenza del processo interno realmente seguito.

In [5]:
# Aggiungere qui altre cartelle di run per un confronto sul medesimo protocollo.
comparison_dirs = list(run_dirs)
if all(result["summary"]["complete_for_comparison"] for result in results.values()):
    comparison = compare_runs(comparison_dirs)
    for row in comparison:
        print(row["model"], "| coppie:", row["pair_success_count"], "/", row["n_pairs"],
              "| score:", row["paired_grounded_revision_score"], "| digest:", row["digest"])
else:
    print("Almeno un run è incompleto: nessuna graduatoria generata.")

qwen2.5-coder:7b | coppie: 3 / 6 | score: 0.5 | digest: dae161e27b0e90dd1856c8bb3209201fd6736d8eb66298e75ed87571486f4364
